# BERT: the main ideas and a hands-on guide

**Paper:** *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding* (Devlin et al., 2018) ? [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)

> **In one sentence:** BERT learns from text by filling in hidden words using context on both sides, then it can be fine-tuned for a task by adding a small output layer.

This notebook has two parts: a focused explanation of the paper, followed by short code experiments you can run and change. We use a pre-trained BERT model rather than building the Transformer from scratch.

# PART A ? Understanding the paper

## 1. Why BERT?

Many language tasks need labeled examples, but unlabeled text is plentiful. **Pre-training** learns general language patterns from plain text; **fine-tuning** adapts that knowledge to a smaller, labeled task.

Before BERT, popular models read in one direction. GPT read left-to-right, while ELMo combined separate left-to-right and right-to-left models. Consider "I went to the **bank** to deposit money" and "I went to the **bank** to go fishing." The clue comes after *bank*, so a model that only reads left-to-right cannot use it when forming the word's representation.

**BERT's key idea:** let each word use both its left and right context at every layer. This is called **deep bidirectionality**. BERT is in the fine-tuning family: the model is first trained on general text, then the whole model is adapted to a particular task.

## 2. A little background

### Tokens and embeddings

A tokenizer splits text into tokens (words or word pieces). Each token becomes a vector of numbers called an **embedding**. Unlike word2vec's fixed vector for "bank," BERT creates a **contextual embedding**: the vector changes with the sentence, so a money-bank and a river-bank can differ.

### Transformer and self-attention

BERT is a stack of Transformer **encoder** layers. In each layer, **self-attention** lets every token gather useful information from other tokens. For example, in "The animal was tired because it ran," attention can help connect *it* to *animal*. Stacking layers builds richer representations.

<details><summary><b>Optional: the attention formula</b></summary>

Each token is represented by a Query, Key, and Value. Attention compares queries with keys, then uses the resulting weights to combine values:

$$	ext{Attention}(Q,K,V)=	ext{softmax}\left(rac{QK^	op}{\sqrt{d_k}}ight)V$$

Multiple heads run this operation in parallel to learn different relationships.
</details>

An **encoder** can use words on both sides and is suited to understanding text (BERT). A **decoder** sees only earlier words, which allows it to generate text one word at a time (GPT).

## 3. BERT's main trick: hide words

A normal language model predicts the next word. A fully bidirectional model could cheat at that task because the answer would already be visible in its input.

BERT instead uses a **Masked Language Model (MLM)**: hide selected words, then predict them from the surrounding context.

> Input: `the cat [MASK] on the mat`  ?  Target: `sat`

This lets BERT use both sides without seeing the answer. The overall process is:

1. **Pre-train once:** learn from large amounts of unlabeled text using MLM and a sentence-pair task.
2. **Fine-tune per task:** add a small task-specific output layer and train BERT on labeled examples.

The Transformer layers stay the same; fine-tuning changes the output layer and adjusts the model weights.

## 4. BERT model sizes

BERT stacks Transformer layers. The paper's model sizes are:

| Model | Layers | Hidden size | Attention heads | Parameters |
|---|---:|---:|---:|---:|
| **BERT-Base** | 12 | 768 | 12 | 110 million |
| **BERT-Large** | 24 | 1024 | 16 | 340 million |

BERT-Base was about the same size as the GPT model used in the paper's comparison, making the comparison fair. BERT-Large is bigger and generally performs better, but also needs more memory and computation.

## 5. How BERT represents its input

BERT uses one format for a single text or a pair of texts. WordPiece breaks uncommon words into smaller pieces (`playing` can become `play`, `##ing`). The model combines three embeddings for each token: its identity, its position, and its sentence segment.

For a pair of texts, the input looks like this:

```text
[CLS] my dog is cute [SEP] he likes playing [SEP]
```

- **`[CLS]`** starts the input. Its final vector is used for many classification tasks.
- **`[SEP]`** marks the end of a text and separates a pair.
- **Segment IDs** mark the first text as A and the second as B.
- **Position embeddings** tell BERT the token order (up to 512 tokens).

The input vectors are the sum of **token + segment + position embeddings**. In the code section, you'll inspect these tokens and segment IDs yourself.

## 6. Pre-training task 1: Masked Language Modeling (MLM)

BERT selects **15% of input tokens** as prediction targets. For each selected token, it predicts the original token from context. To avoid relying only on `[MASK]` tokens (which do not appear in ordinary text), BERT changes selected tokens this way:

| Portion of selected tokens | Input change | Example target |
|---:|---|---|
| 80% | Replace with `[MASK]` | `my dog is [MASK]` ? `hairy` |
| 10% | Replace with a random token | `my dog is apple` ? `hairy` |
| 10% | Leave unchanged | `my dog is hairy` ? `hairy` |

Only selected positions contribute to the MLM prediction loss. This means fewer targets per training example than next-word prediction, but the model learns from context on both sides.

## 7. Pre-training task 2: Next Sentence Prediction (NSP)

Some tasks depend on how two texts relate, so the original BERT paper added **NSP**. BERT receives text A and text B and predicts whether B really followed A in the source document:

- **50% IsNext:** B is the actual following text.
- **50% NotNext:** B is a random text.

The model makes this prediction from the `[CLS]` vector, while MLM runs at the same time. Later work (for example, RoBERTa) found that NSP can be removed when training on more data for longer; it is still useful to know because it was part of the original BERT recipe.

One caveat: raw pre-trained `[CLS]` is not automatically a good general-purpose sentence embedding. For similarity tasks, use a model trained specifically for sentence embeddings or fine-tune BERT.

## 8. Pre-training data and cost

The paper trained BERT on **BooksCorpus and English Wikipedia** (about 3.3 billion words). The data contains long passages so BERT can learn from sentence context and sentence pairs. Pre-training used large batches and TPU hardware; BERT-Base and BERT-Large each took about four days.

Pre-training is expensive, but it is done once. The released weights let others download BERT and spend much less time fine-tuning it for their own tasks.

## 9. Fine-tuning: use BERT for a task

To fine-tune, start with pre-trained BERT, add a small output layer, and train **all** the weights on labeled examples. The input format stays the same across tasks:

| Task | Input | What the output layer uses |
|---|---|---|
| Sentiment or topic classification | `[CLS] text [SEP]` | `[CLS]` vector |
| Paraphrase or inference | `[CLS] A [SEP] B [SEP]` | `[CLS]` vector |
| Question answering | `[CLS] question [SEP] passage [SEP]` | Passage token vectors for answer start/end |
| Name/entity tagging | `[CLS] text [SEP]` | Each token's vector |

The original paper recommends a small learning rate (about `2e-5` to `5e-5`) and only **2?4 epochs**. Fine-tuning usually takes minutes or hours, rather than the days needed for pre-training.

## 10. Results: what did the paper show?

BERT reported new best results on **11 language-understanding tasks**. Examples include GLUE increasing from 72.8 to 80.5 and SQuAD question-answering F1 reaching 93.2.

The paper also ran **ablations**: remove one part of the training recipe and see what changes. Removing NSP had a smaller effect, while replacing bidirectional training with left-to-right training hurt several tasks, especially question answering. This supports the paper's main claim that using context from both sides is important. Larger BERT models also improved results, including on relatively small datasets.

**Main takeaway:** pre-training gives BERT broad language knowledge; bidirectional context helps it represent each word; fine-tuning adapts those representations to a specific task.

## Recap and quick check

1. BERT is pre-trained on plain text, then fine-tuned for a task.
2. It reads both left and right context in every encoder layer.
3. MLM hides selected tokens so the model cannot copy the answer.
4. Original BERT also predicts whether one text follows another (NSP).
5. `[CLS]` is commonly used for classification; token vectors are used for tagging and answer spans.

Try to answer before opening each explanation:

<details><summary><b>Why does BERT hide tokens?</b></summary>So it can use both sides of a word without seeing the answer it must predict.</details>

<details><summary><b>What does the classifier use for sentiment?</b></summary>The final `[CLS]` vector.</details>

<details><summary><b>Can BERT generate a story like GPT?</b></summary>Not naturally. BERT is trained to understand text; GPT's left-to-right setup is designed for generating it.</details>

# PART B ? Try BERT with code

The following experiments show how BERT tokenizes text, fills in a blank, predicts sentence relationships, creates contextual word vectors, and can be fine-tuned for classification. Each section has a small code example and a suggestion to change it.

**Setup:** Python 3.9+ and an internet connection. The first run downloads the model (about 440 MB). A CPU is enough; a GPU is optional. Run cells from top to bottom. If installing or changing package versions, restart the kernel and run the cells again.

In [2]:
!pip install -q "transformers<5" torch


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\dbhagria\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [9]:
import torch
from transformers import AutoTokenizer, pipeline

MODEL_NAME = "bert-base-uncased"   # the original BERT-Base from the paper (lowercased English)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)   # the paper says ~30,000 WordPiece tokens

c:\Users\dbhagria\AppData\Local\miniconda3\envs\assignments\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1116 13:12:18.947000 15512 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\dbhagria\AppData\Local\miniconda3\envs\assignments\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dbhagria\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, 

Vocabulary size: 30522


## 12. Inspect the input format

Tokenize a pair of short texts and view the tokens, their positions, and their sentence IDs. This shows how BERT adds `[CLS]`, `[SEP]`, and WordPiece tokens.

In [10]:
encoded = tokenizer("my dog is cute", "he likes playing")

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print(f"{'position':<10}{'token':<10}{'segment':<8}")
for pos, (tok, seg) in enumerate(zip(tokens, encoded["token_type_ids"])):
    print(f"{pos:<10}{tok:<10}{'A' if seg == 0 else 'B':<8}")

position  token     segment 
0         [CLS]     A       
1         my        A       
2         dog       A       
3         is        A       
4         cute      A       
5         [SEP]     A       
6         he        B       
7         likes     B       
8         playing   B       
9         [SEP]     B       


The output shows the special tokens, WordPiece splitting, and segment IDs: `0` marks text A and `1` marks text B. **Try it:** replace the example texts or tokenize a rare word.

In [11]:
print(tokenizer.tokenize("Transformers are unbelievably good at tokenization"))

['transformers', 'are', 'un', '##bel', '##ie', '##va', '##bly', 'good', 'at', 'token', '##ization']


## 13. Fill in a blank (MLM)

The `fill-mask` pipeline uses BERT's pre-trained MLM head to suggest a token for `[MASK]`.

In [12]:
fill_mask = pipeline("fill-mask", model=MODEL_NAME)

def show_guesses(sentence, k=5):
    print(sentence)
    for guess in fill_mask(sentence, top_k=k):
        print(f"   {guess['token_str']:<12} {guess['score']:.3f}")
    print()

show_guesses("The capital of France is [MASK].")
show_guesses("My dog is [MASK].")   # the paper's own example sentence

c:\Users\dbhagria\AppData\Local\miniconda3\envs\assignments\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dbhagria\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1172.11it/s]
[transformers] BertForMaskedLM LOAD REPORT fro

The capital of France is [MASK].
   paris        0.417
   lille        0.071
   lyon         0.063
   marseille    0.044
   tours        0.030

My dog is [MASK].
   dead         0.131
   here         0.096
   fine         0.080
   gone         0.069
   missing      0.036



### Test why both sides matter

These sentences have the same words before `[MASK]` but different clues after it. Compare BERT's guesses to see how right-hand context changes its prediction.

In [13]:
show_guesses("I went to the [MASK] to deposit my paycheck.")
show_guesses("I went to the [MASK] to see the doctor.")
show_guesses("I went to the [MASK] to buy some bread.")

I went to the [MASK] to deposit my paycheck.
   bank         0.901
   office       0.009
   store        0.008
   teller       0.008
   atm          0.006

I went to the [MASK] to see the doctor.
   hospital     0.693
   bathroom     0.052
   office       0.025
   clinic       0.025
   doctor       0.015

I went to the [MASK] to buy some bread.
   market       0.780
   store        0.056
   bakery       0.047
   kitchen      0.023
   markets      0.015



**Try it:** write your own sentence with one `[MASK]`. Put a useful clue after the blank and see what changes. BERT fills one token; it is not a text generator.

In [18]:
show_guesses("alice was born in [MASK], of Texas.")

alice was born in [MASK], of Texas.
   houston      0.276
   dallas       0.270
   austin       0.105
   beaumont     0.019
   texas        0.011



## 14. Predict whether one text follows another (NSP)

The original BERT checkpoint includes an NSP head. It reads a pair of texts and predicts **IsNext** or **NotNext** from the `[CLS]` representation.

In [19]:
from transformers import BertForNextSentencePrediction

nsp_model = BertForNextSentencePrediction.from_pretrained(MODEL_NAME)
nsp_model.eval()

def is_next_probability(sentence_a, sentence_b):
    inputs = tokenizer(sentence_a, sentence_b, return_tensors="pt")
    with torch.no_grad():
        logits = nsp_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    return probs[0].item()   # index 0 = "IsNext", index 1 = "NotNext"

a = "The man went to the store."
print(f"{is_next_probability(a, 'He bought a gallon of milk.'):.3f}  <- real continuation")
print(f"{is_next_probability(a, 'Penguins are flightless birds.'):.3f}  <- random sentence")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1121.85it/s]
[transformers] BertForNextSentencePrediction LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1.000  <- real continuation
0.000  <- random sentence


The first pair is a real continuation; the second is unrelated. Compare the model's IsNext probabilities.

## 15. Contextual word vectors

BERT gives a word a vector based on its context. The example compares "bank" in two money sentences and one river sentence using cosine similarity.

In [20]:
from transformers import AutoModel
import torch.nn.functional as F

bert = AutoModel.from_pretrained(MODEL_NAME)   # BERT body only, no task head
bert.eval()

def word_vector(sentence, word):
    # Return BERT's final-layer vector for `word` inside `sentence`.
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = bert(**inputs).last_hidden_state[0]      # shape: (num_tokens, 768)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return hidden[tokens.index(word)]

money_1 = word_vector("I deposited cash at the bank.", "bank")
money_2 = word_vector("The bank approved my loan application.", "bank")
river   = word_vector("We had a picnic on the bank of the river.", "bank")

sim = lambda x, y: F.cosine_similarity(x, y, dim=0).item()
print(f"money bank  vs money bank : {sim(money_1, money_2):.3f}")
print(f"money bank  vs river bank : {sim(money_1, river):.3f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1071.31it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


money bank  vs money bank : 0.828
money bank  vs river bank : 0.512


The two money-related uses should be more similar than the river use. **Try it:** change the sentences to another ambiguous word, such as `bat` or `spring`.

## 16. Use a fine-tuned BERT model

Many BERT models are already fine-tuned and ready to use. The next examples show question answering (which selects an answer span from a passage) and sentiment classification.

In [ ]:
qa = pipeline("question-answering", model="deepset/bert-base-cased-squad2")

passage = (
    "BERT was introduced by researchers at Google AI Language in 2018. "
    "It was pre-trained on BooksCorpus and English Wikipedia, which together contain "
    "about 3.3 billion words. Pre-training BERT-Large took four days on 64 TPU chips."
)

for question in ["Who introduced BERT?",
                 "How many words were in the training data?",
                 "How long did pre-training take?"]:
    result = qa(question=question, context=passage)
    print(f"Q: {question}\nA: {result['answer']}   (score {result['score']:.2f}, "
          f"characters {result['start']}-{result['end']})\n")

The answer is copied from the passage: BERT predicts its start and end positions rather than writing new text.

### Sentiment classification

This model has been fine-tuned to predict a 1?5 star rating from a review.

In [ ]:
sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

reviews = [
    "Absolutely loved it, would buy again!",
    "It's okay, nothing special.",
    "Broke after two days. Total waste of money.",
    "Ce produit est excellent.",              # French - it's a multilingual BERT
]
for review, pred in zip(reviews, sentiment(reviews)):
    print(f"{pred['label']:<8} ({pred['score']:.2f})  {review}")

## 17. Fine-tune a small classifier

This toy example trains BERT to label sentences as **food** or **sports**. It follows the real workflow: load pre-trained BERT, add a classification layer, tokenize examples, and train on labels.

The new classifier layer starts with random weights, so a warning about uninitialized classifier weights is expected. The tiny dataset is only for learning the steps; real projects need far more examples.

In [ ]:
from transformers import AutoModelForSequenceClassification

torch.manual_seed(0)

# Toy task: is a sentence about FOOD (1) or SPORTS (0)?
train_texts = [
    "The pasta was cooked perfectly with fresh basil.",  "She scored two goals in the final match.",
    "This soup needs more salt and pepper.",             "The team won the championship last night.",
    "I baked a chocolate cake for dessert.",             "He ran the marathon in under three hours.",
    "The sushi at that restaurant is amazing.",          "The coach called a timeout in the last quarter.",
    "Grilled vegetables taste great with olive oil.",    "Our goalkeeper saved a penalty kick.",
    "We ordered spicy noodles and dumplings.",           "The tennis player served an ace to win the set.",
    "Fresh bread smells wonderful in the morning.",      "The referee showed him a red card.",
    "This curry is rich and full of flavor.",            "She broke the world record in swimming.",
]
train_labels = [1, 0] * 8

classifier = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

In [ ]:
optimizer = torch.optim.AdamW(classifier.parameters(), lr=5e-5)   # a learning rate from the paper's range
batch = tokenizer(train_texts, padding=True, return_tensors="pt")
labels = torch.tensor(train_labels)

classifier.train()
for epoch in range(10):
    outputs = classifier(**batch, labels=labels)   # forward pass; the library computes the loss
    outputs.loss.backward()                        # compute gradients for ALL weights (BERT + W)
    optimizer.step()                               # update the weights
    optimizer.zero_grad()
    print(f"epoch {epoch + 1:>2}   loss {outputs.loss.item():.4f}")

The loss should generally decrease during training. Now test on sentences not used for training. With only a few examples, predictions are just a demonstration?not a reliable classifier.

In [ ]:
test_texts = [
    "I had a delicious burrito for lunch.",
    "The striker missed an easy shot.",
    "Mix the flour, sugar, and eggs in a bowl.",
    "The cyclists raced up the mountain.",
]

classifier.eval()
with torch.no_grad():
    probs = torch.softmax(classifier(**tokenizer(test_texts, padding=True, return_tensors="pt")).logits, dim=1)

for text, p in zip(test_texts, probs):
    label = "FOOD" if p[1] > 0.5 else "SPORTS"
    print(f"{label:<7} ({max(p).item():.2f})  {text}")

## Ideas to explore next

- Replace the toy training sentences with a real dataset and compare results.
- Compare vectors from different BERT layers.
- Try a different pre-trained model or task.

**Related models:** RoBERTa removes NSP and trains longer; DistilBERT is smaller and faster; Sentence-BERT is designed for sentence similarity.

**Further reading:** [BERT paper](https://arxiv.org/abs/1810.04805) ? [Original code and weights](https://github.com/google-research/bert) ? [The Illustrated BERT](https://jalammar.github.io/illustrated-bert/)